# Section 1 : Environment Setup & Imports
Installing the required dependencies and load API keys.

## 1.1 installing dependencies

In [1]:
import warnings
warnings.filterwarnings('ignore')

# !pip install -r requirements.txt

## 1.2 Loading the OpenAI API key

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Section 2 : Data Chunking
loading the mock data and split it into chunks.

In [3]:
from rag_utils import DataProcessor

# Initialize the Data processor
processor = DataProcessor()

# Load and split the document
chunks = processor.load_and_split(directory_path="data/", chunk_size=500, chunk_overlap=100)

print(f"Total chunks created: {len(chunks)}")

sources = set([chunk.metadata.get("source") for chunk in chunks])
print(f"Sources loaded: {sources}")

100%|██████████| 1/1 [00:00<00:00, 307.68it/s]

Total chunks created: 44
Sources loaded: {'data/github_projects.md', 'data/Mohammad_CV.pdf', 'data/linkedin_profile.txt', 'data/certifications.txt'}


# Section 3 : Vector Database & Retriever
Convert the text chunks into embeddings using HuggingFace's `all-MiniLM-L6-v2` model and store them in a ChromaDB vector store.

In [4]:
# Create vector store
vectorstore = processor.create_vector_store(chunks, persist_directory="./chroma_db")
# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Section 4 : Building the RAG Pipeline
inject our retriever into the `RAGPipeline` class. This class handles the prompt engineering and the LangChain Expression Language (LCEL) to ensure the LLM answers based *only* on the retrieved context and cites its sources.

In [5]:
from rag_chain import RAGPipeline

# Initialize the RAG pipeline with the retriever
rag_pipeline = RAGPipeline(retriever=retriever)
rag_chain = rag_pipeline.build()

# Section 5 : Inference

In [6]:
def query_rag(question: str):
    print(f"Question: {question}\n")

    # Invoke the chain
    result = rag_chain.invoke(question)

    print(f"Answer:\n{result['answer']}\n")

    print("Source Documents Used:")
    for i, doc in enumerate(result["context"]):
        # Print the source file name along with the content
        source_name = doc.metadata.get("source", "Unknown")
        print(f"  [Source {i+1} - {source_name}]:")
        print(f"  {doc.page_content.strip()}\n")
    print("-" * 50)

## Test 1: Specific skill question (Tests Skills & Languages section)

In [7]:
query_rag("What are Mohammad's main programming languages?")

Question: What are Mohammad's main programming languages?

Answer:
Mohammad's main programming languages are Python, Swift, C++, Java, and Dart.

Source Documents Used:
  [Source 1 - data/certifications.txt]:
CERTIFICATION SUMMARY
Mohammad's formal development history includes data science and machine
learning training, artificial-intelligence training, the PIP professional
development curriculum, Flutter and Dart education, and native iOS training.

  [Source 2 - data/certifications.txt]:
  Notes:
Mohammad completed this program as part of his transition toward practical
machine-learning and AI work. The program focused on applying data science
concepts to technical projects rather than only studying theory.

  [Source 3 - data/linkedin_profile.txt]:
  Location:
Jenin, Palestine

Study Period:
June 2019 – April 2025

GPA:
3.21 / 4.0

Academic Areas:
- Algorithms
- Data structures
- Object-oriented programming
- Software engineering
- Computer systems
- Networking
- Databases
- Artific

## Test 2: Experience question (Tests Professional Experience section)

In [8]:
query_rag("Where did Mohammad work and what did he do there?")

Question: Where did Mohammad work and what did he do there?

Answer:
Mohammad works as an AI Engineer Intern at Cellula Technologies, a remote position he has held since September 2026. In this role, he focuses on AI engineering tasks as part of his internship.

Source Documents Used:
  [Source 1 - data/certifications.txt]:
CERTIFICATION SUMMARY
Mohammad's formal development history includes data science and machine
learning training, artificial-intelligence training, the PIP professional
development curriculum, Flutter and Dart education, and native iOS training.

  [Source 2 - data/certifications.txt]:
  Notes:
Mohammad completed this program as part of his transition toward practical
machine-learning and AI work. The program focused on applying data science
concepts to technical projects rather than only studying theory.

  [Source 3 - data/linkedin_profile.txt]:
  ## PROFESSIONAL EXPERIENCE

### AI Engineer Intern — Cellula Technologies
Employment Period: September 2026 – Present
W

## Test 3: GitHub question (Tests Projects section)

In [9]:
query_rag("What kind of projects has Mohammad worked on?")

Question: What kind of projects has Mohammad worked on?

Answer:
Based on the provided context, Mohammad has worked on projects that involve applying data science and machine‑learning concepts to technical work, as part of his transition toward practical ML and AI. His training covered data science, machine learning, artificial intelligence, the PIP professional development curriculum, Flutter and Dart education, and native iOS development. In his current role as an AI Engineer Intern at Cellula Technologies (remote), he is engaged in AI‑focused projects. The context does not list specific project names, so the answer reflects the types of projects mentioned.

Source Documents Used:
  [Source 1 - data/certifications.txt]:
CERTIFICATION SUMMARY
Mohammad's formal development history includes data science and machine
learning training, artificial-intelligence training, the PIP professional
development curriculum, Flutter and Dart education, and native iOS training.

  [Source 2 - data/cer

## Test 4: Certifications question (Tests Certifications section)

In [10]:
query_rag("What certifications does Mohammad have?")

Question: What certifications does Mohammad have?

Answer:
Based on the certification record provided, Mohammad has completed the following certifications and professional‑development programs:

- **Data Science & Machine Learning** – offered by IBT Learning Solutions & GGateway (completed 03 Jan 2026 – 15 Apr 2026, 3 months, status: Completed)  
- Artificial‑intelligence training  
- PIP professional development curriculum  
- Flutter and Dart education  
- Native iOS training  

These are the certifications and training programs listed in the context.

Source Documents Used:
  [Source 1 - data/certifications.txt]:
CERTIFICATION SUMMARY
Mohammad's formal development history includes data science and machine
learning training, artificial-intelligence training, the PIP professional
development curriculum, Flutter and Dart education, and native iOS training.

  [Source 2 - data/certifications.txt]:
CERTIFICATION RECORD 01
Program Name: Data Science & Machine Learning
Provider: IBT Learni

## Test 5: Out-of-scope question (testing hallucination guardrails)

In [11]:
query_rag("What is Mohammad's favorite food?")

Question: What is Mohammad's favorite food?

Answer:
I don't have enough information to answer that.

Source Documents Used:
  [Source 1 - data/certifications.txt]:
CERTIFICATION SUMMARY
Mohammad's formal development history includes data science and machine
learning training, artificial-intelligence training, the PIP professional
development curriculum, Flutter and Dart education, and native iOS training.

  [Source 2 - data/certifications.txt]:
  Notes:
Mohammad completed this program as part of his transition toward practical
machine-learning and AI work. The program focused on applying data science
concepts to technical projects rather than only studying theory.

  [Source 3 - data/linkedin_profile.txt]:
  ## PROFESSIONAL EXPERIENCE

### AI Engineer Intern — Cellula Technologies
Employment Period: September 2026 – Present
Work Arrangement: Remote

Role Focus:
Mohammad works as an AI Engineer Intern at Cellula Technologies.

--------------------------------------------------


In [12]:
query_rag("What is Mohammad's summmary in his cv ?")

Question: What is Mohammad's summmary in his cv ?

Answer:
Mohammad's formal development history includes data science and machine learning training, artificial‑intelligence training, the PIP professional development curriculum, Flutter and Dart education, and native iOS training.

Source Documents Used:
  [Source 1 - data/certifications.txt]:
CERTIFICATION SUMMARY
Mohammad's formal development history includes data science and machine
learning training, artificial-intelligence training, the PIP professional
development curriculum, Flutter and Dart education, and native iOS training.

  [Source 2 - data/certifications.txt]:
CERTIFICATION RECORD 01
Program Name: Data Science & Machine Learning
Provider: IBT Learning Solutions & GGateway
Category: Data Science / Machine Learning
Start Date: 03 January 2026
End Date: 15 April 2026
Duration: 3 months
Status: Completed
Recognition: Professional training program
Notes:
Mohammad completed this program as part of his transition toward practica